Step 1: Install, bootstrap instructions

In [ ]:
!pip install --quiet anthropic pydantic

!rm -rf /content/astra-swarm 2>/dev/null
!git clone --depth 1 -q https://github.com/phdeore/astra-swarm.git /content/astra-swarm

import sys
sys.path.insert(0, "/content/astra-swarm/src")

# Verify before importing
import os
pkg = "/content/astra-swarm/src/astra_swarm"
assert os.path.isdir(pkg), f"missing folder: {pkg}"
assert os.path.isfile(f"{pkg}/__init__.py"), f"missing: {pkg}/__init__.py"
assert os.path.isfile(f"{pkg}/schemas.py"), f"missing: {pkg}/schemas.py"

from astra_swarm.schemas import Incident
print("astra-swarm source loaded")

Step 2: Load key from Collab Secrets, create client

In [ ]:
from google.colab import userdata
from anthropic import Anthropic
import os

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
client = Anthropic()
print("Client ready to use!")

Step 3: Message call invocation

In [ ]:
from anthropic import APIStatusError, BadRequestError

MODEL = "claude-haiku-4-5-20251001"
MAX_TOKENS = 512

try:
    response = client.messages.create(
        model=MODEL,
        max_tokens=MAX_TOKENS,
        messages=[
            {"role": "user", "content": "In one sentence, what is a Security Operations Center?"}
        ]
    )
    print("API call successful:")
    print(response.content[0].text)
except BadRequestError as e:
    print(f"Bad Request Error: {e.response.json()['error']['message']}")
    print("Please check your API request parameters or your Anthropic account's billing status.")
except APIStatusError as e:
    print(f"An API error occurred: {e.status_code} - {e.response.json()['error']['message']}")
    print("You might need to check your internet connection or the Anthropic API service status.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Step 4: Understand the response

In [ ]:
print("model:", response.model)
print("stop reason:", response.stop_reason)
print("input tokens:", response.usage.input_tokens)
print("output tokens:", response.usage.output_tokens)
print("-" * 40)

for i, block in enumerate(response):
    if block[0] == "content":
        print(i, block)
        print("-" * 40)

Step 5: Helper function

In [ ]:
def ask(prompt: str, system: str | None = None, max_tokens: int = 512) -> str:
    """Send one user message to Claude and return the concatenated text."""
    kwargs = {
        "model": MODEL,
        "max_tokens": MAX_TOKENS,
        "messages": [{"role": "user", "content": prompt}]
    }
    if system:
        kwargs["system"] = system
    response = None
    try:
        response = client.messages.create(**kwargs)
        print(response)
    except Anthropic.APIStatusError as e:
        print(f"An API error occurred: {e.status_code} - {e.response.json()['error']['message']}")
        print("You might need to check your internet connection or the Anthropic API service status.")
    result = "".join(r.text for r in response.content if r.type == "text")
    return result.replace("\n", "")

# test helper function
ask("Give one liner definition of MITRE ATT&CK.")

Step 6: Smoke test of schema

In [ ]:
from astra_swarm.schemas import Incident, AlertClass, Severity, IdentitySignals, ProposedAction

demo = Incident(
    alert_class=AlertClass.IDENTITY_AUTH,
    entities_users=["alice@example.com"],
    entities_ips=["203.0.113.7"],
    attack_techniques=["T1078"],  # Valid Accounts
    severity=Severity.HIGH,
    severity_rationale="Login from a country the user has never logged in from, "
                       "5 min after a successful login from their normal geo.",
    identity_signals=IdentitySignals(impossible_travel=True,
                                     notes="Delta between logins < 5 min, 8000 km apart."),
    confidence=0.82,
    recommended_response="Force reauth, notify user via out-of-band channel, "
                         "temporarily require step-up MFA on all sessions.",
    proposed_actions=[
        ProposedAction(action="force_reauth",
                       target="alice@example.com",
                       rationale="Suspected credential theft — impossible travel."),
    ],
)
print(demo.model_dump_json(indent=2))

In [ ]:
# Response for the above incident from the LLM

{
  "incident_id": "6fb0810d-889a-4e94-9262-7d5ef34b7ef8",
  "created_at": "2026-08-13T07:05:42.769044Z",
  "source_alert_ids": [],
  "alert_class": "identity_auth",
  "entities_users": [
    "alice@example.com"
  ],
  "entities_hosts": [],
  "entities_ips": [
    "203.0.113.7"
  ],
  "attack_techniques": [
    "T1078"
  ],
  "enrichment_notes": "",
  "correlation_notes": "",
  "identity_signals": {
    "impossible_travel": true,
    "mfa_fatigue": false,
    "privilege_escalation": false,
    "dormant_account_reactivation": false,
    "notes": "Delta between logins < 5 min, 8000 km apart."
  },
  "severity": "high",
  "severity_rationale": "Login from a country the user has never logged in from, 5 min after a successful login from their normal geo.",
  "confidence": 0.82,
  "recommended_response": "Force reauth, notify user via out-of-band channel, temporarily require step-up MFA on all sessions.",
  "proposed_actions": [
    {
      "action": "force_reauth",
      "target": "alice@example.com",
      "rationale": "Suspected credential theft — impossible travel.",
      "requires_approval": true
    }
  ],
  "requires_human_approval": true
}
